# Electoral Gender Quotas and Democratic Legitimacy

**Authors:** Amanda Clayton, Diana Z. O'Brien, Jennifer M. Piscopo

**Journal:** American Political Science Review (2026)

This notebook reproduces the analysis from the paper above.
It was auto-generated by [PoliRep](https://polirep.org).

---

## Setup

Install packages and load standard libraries.

In [ ]:
install.packages(c("tidyverse", "fixest", "modelsummary", "broom", "lmtest", "sandwich", "stargazer"))

library(tidyverse)
library(fixest)

## Data Preparation


### Load Country-Level Aggregated Data

Load all 10 country-level aggregated datasets (ArgAgg, BrazilAgg,
MexAgg, PeruAgg, NZAgg, USAgg, UKAgg, SpainAgg, PortAgg, NorAusFranceAgg).
Each file contains pre-computed legitimacy indices and cleaned demographic
variables.

In [ ]:
# Load and merge country-level datasets

cat("Loading country-level aggregated datasets...\n")

# Read in country level datasets
MexAgg <- read.csv(file.path(data_dir, "MexAgg.csv"), row.names = 1)
PeruAgg <- read.csv(file.path(data_dir, "PeruAgg.csv"), row.names = 1)
UKAgg <- read.csv(file.path(data_dir, "UKAgg.csv"), row.names = 1)
SpainAgg <- read.csv(file.path(data_dir, "SpainAgg.csv"), row.names = 1)
PortAgg <- read.csv(file.path(data_dir, "PortAgg.csv"), row.names = 1)
USAgg <- read.csv(file.path(data_dir, "USAgg.csv"), row.names = 1)
BrazilAgg <- read.csv(file.path(data_dir, "BrazilAgg.csv"), row.names = 1)
ArgAgg <- read.csv(file.path(data_dir, "ArgAgg.csv"), row.names = 1)
NZAgg <- read.csv(file.path(data_dir, "NZAgg.csv"), row.names = 1)
NorAusFranceAgg <- read.csv(file.path(data_dir, "NorAusFranceAgg.csv"), row.names = 1)

cat("Loaded", nrow(MexAgg), "rows from Mexico\n")
cat("Loaded", nrow(PeruAgg), "rows from Peru\n")
cat("Loaded", nrow(UKAgg), "rows from UK\n")
cat("Loaded", nrow(SpainAgg), "rows from Spain\n")
cat("Loaded", nrow(PortAgg), "rows from Portugal\n")
cat("Loaded", nrow(USAgg), "rows from USA\n")
cat("Loaded", nrow(BrazilAgg), "rows from Brazil\n")
cat("Loaded", nrow(ArgAgg), "rows from Argentina\n")
cat("Loaded", nrow(NZAgg), "rows from NZ\n")
cat("Loaded", nrow(NorAusFranceAgg), "rows from Norway/Australia/France\n")

# Make sure that names are the same across country level datasets to prepare for merging
names(UKAgg) <- names(MexAgg)
names(SpainAgg) <- names(MexAgg)
names(PortAgg) <- names(MexAgg)
names(BrazilAgg) <- names(MexAgg)
names(ArgAgg) <- names(MexAgg)
names(NZAgg) <- names(MexAgg)
names(NorAusFranceAgg) <- names(MexAgg)

# Merge (rbind) all datasets
MergedN <- rbind(MexAgg, PeruAgg, UKAgg, SpainAgg, PortAgg, USAgg, BrazilAgg, ArgAgg, NZAgg, NorAusFranceAgg)

cat("Merged dataset has", nrow(MergedN), "rows and", ncol(MergedN), "columns\n")

In [ ]:
source("r_clean/scripts/00_setup.R")
source("r_clean/scripts/02_load_data.R")

cat("Dimensions:", nrow(MergedN), "x", ncol(MergedN), "\n")
cat("Countries:\n")
print(table(MergedN$Country))
cat("\nTreatment distribution:\n")
print(table(MergedN$treat))
head(MergedN)


### Recode Treatment Variable

Convert text treatment labels to numeric codes. UK/Spain/Portugal use
"treat1"..."treat6" labels; other countries use numeric 1-6. Standardize
to numeric 1-6 for all countries.

In [ ]:
# Ensure treatment numbers are the same across datasets
MergedN$treat <- recode(MergedN$treat, "'treat1' = 1; 'treat2' = 2; 'treat3' = 3; 'treat4' = 4; 'treat5' = 5; 'treat6' = 6")

In [ ]:
source("r_clean/scripts/00_setup.R")
source("r_clean/scripts/02_load_data.R")

cat("Treatment coding (after recode):\n")
print(table(MergedN$treat))
cat("\nTreatment class:", class(MergedN$treat), "\n")
cat("Missing treatments:", sum(is.na(MergedN$treat)), "\n")


### Recode Gender and Ideology

Recode gender from language-specific text labels to binary (1=Female, 0=Male).
Recode ideology from text labels to numeric 0-10 scale. Apply country-specific
mappings from config.

In [ ]:
# Recode gender variables for consistency across datasets
MergedN[MergedN == "Female"] <- 1
MergedN[MergedN == "Feminino"] <- 1
MergedN[MergedN == "Mujer"] <- 1
MergedN[MergedN == "Woman"] <- 1
MergedN[MergedN == "Hombre"] <- 0
MergedN[MergedN == "Male"] <- 0
MergedN[MergedN == "Man"] <- 0
MergedN[MergedN == "Masculino"] <- 0
MergedN[MergedN == "Other"] <- NA
MergedN[MergedN == "Otro/ No binario"] <- NA

# Recode values on political ideology, for consistency across datasets
MergedN[MergedN == "Left\n(0)"] <- 0
MergedN[MergedN == "Right\n(10)"] <- 10

MergedN$leftright_1 <- as.numeric(as.character(MergedN$leftright_1))
MergedN$gender <- as.numeric(as.character(MergedN$gender))

In [ ]:
source("r_clean/scripts/00_setup.R")
source("r_clean/scripts/02_load_data.R")

cat("Gender distribution:\n")
print(table(MergedN$gender, useNA = "ifany"))
cat("\nIdeology summary:\n")
print(summary(MergedN$leftright_1))
cat("\nIdeology missing:", sum(is.na(MergedN$leftright_1)),
    sprintf("(%.1f%%)", 100 * mean(is.na(MergedN$leftright_1))), "\n")


### Compute Sample Statistics

Calculate sample size, country distribution, treatment balance, and
legitimacy outcome completeness. Report missing data patterns.

In [ ]:
# Merge (rbind) all datasets
MergedN <- rbind(MexAgg, PeruAgg, UKAgg, SpainAgg, PortAgg, USAgg, BrazilAgg, ArgAgg, NZAgg, NorAusFranceAgg)

cat("Merged dataset has", nrow(MergedN), "rows and", ncol(MergedN), "columns\n")

In [ ]:
source("r_clean/scripts/00_setup.R")
source("r_clean/scripts/02_load_data.R")

cat("Total observations:", nrow(MergedN), "\n")
cat("Countries:", length(unique(MergedN$Country)), "\n")
cat("Complete SubLegStand:", sum(!is.na(MergedN$SubLegStand)), "\n")
cat("Complete ProLegStand:", sum(!is.na(MergedN$ProLegStand)), "\n")

cat("\nCountry distribution:\n")
print(sort(table(MergedN$Country)))

cat("\nTreatment distribution:\n")
print(table(MergedN$treat))


### Create Treatment Subsets

Split the merged dataset into sexual harassment vignettes (treatments 1-3)
and animal mistreatment vignettes (treatments 4-6) for separate analysis.
Australia/Norway/France only have harassment vignettes.

In [ ]:
# Subset dataset for those who saw a vignette about sexual harassment
harass <- subset(MergedN, treat == 1 | treat == 2 | treat == 3)
cat("Sexual harassment subset:", nrow(harass), "rows\n")

# Subset dataset for those who saw animal mistreatment vignette
animal <- subset(MergedN, treat == 4 | treat == 5 | treat == 6)
cat("Animal mistreatment subset:", nrow(animal), "rows\n")

In [ ]:
source("r_clean/scripts/00_setup.R")
source("r_clean/scripts/02_load_data.R")

cat("Sexual harassment sample:", nrow(harass), "obs\n")
cat("  Treatment 1 (All-Male):", sum(harass$treat == 1), "\n")
cat("  Treatment 2 (Gender-Balanced):", sum(harass$treat == 2), "\n")
cat("  Treatment 3 (Quota):", sum(harass$treat == 3), "\n")

cat("\nAnimal mistreatment sample:", nrow(animal), "obs\n")
cat("  Treatment 4 (All-Male):", sum(animal$treat == 4), "\n")
cat("  Treatment 5 (Gender-Balanced):", sum(animal$treat == 5), "\n")
cat("  Treatment 6 (Quota):", sum(animal$treat == 6), "\n")

cat("\nCountries in harassment sample:", length(unique(harass$Country)), "\n")
cat("Countries in animal sample:", length(unique(animal$Country)), "\n")


## 1. Country-Level Data Processing


### 1. Country-Level Data Processing

Load and process 10 country datasets:
- Load raw survey data for each country
- Create factor-analyzed legitimacy indices (omega method)
- Standardize outcomes within country (mean=0, SD=1)
- Filter manipulation check failures and speedsters
- Output aggregated country files

Countries: Argentina, Australia, Brazil, France, Mexico, New Zealand,
Norway, Peru, Portugal, Spain, UK, USA

In [ ]:
# Load and merge country-level datasets

cat("Loading country-level aggregated datasets...\n")

# Read in country level datasets
MexAgg <- read.csv(file.path(data_dir, "MexAgg.csv"), row.names = 1)
PeruAgg <- read.csv(file.path(data_dir, "PeruAgg.csv"), row.names = 1)
UKAgg <- read.csv(file.path(data_dir, "UKAgg.csv"), row.names = 1)
SpainAgg <- read.csv(file.path(data_dir, "SpainAgg.csv"), row.names = 1)
PortAgg <- read.csv(file.path(data_dir, "PortAgg.csv"), row.names = 1)
USAgg <- read.csv(file.path(data_dir, "USAgg.csv"), row.names = 1)
BrazilAgg <- read.csv(file.path(data_dir, "BrazilAgg.csv"), row.names = 1)
ArgAgg <- read.csv(file.path(data_dir, "ArgAgg.csv"), row.names = 1)
NZAgg <- read.csv(file.path(data_dir, "NZAgg.csv"), row.names = 1)
NorAusFranceAgg <- read.csv(file.path(data_dir, "NorAusFranceAgg.csv"), row.names = 1)

cat("Loaded", nrow(MexAgg), "rows from Mexico\n")
cat("Loaded", nrow(PeruAgg), "rows from Peru\n")
cat("Loaded", nrow(UKAgg), "rows from UK\n")
cat("Loaded", nrow(SpainAgg), "rows from Spain\n")
cat("Loaded", nrow(PortAgg), "rows from Portugal\n")
cat("Loaded", nrow(USAgg), "rows from USA\n")
cat("Loaded", nrow(BrazilAgg), "rows from Brazil\n")
cat("Loaded", nrow(ArgAgg), "rows from Argentina\n")
cat("Loaded", nrow(NZAgg), "rows from NZ\n")
cat("Loaded", nrow(NorAusFranceAgg), "rows from Norway/Australia/France\n")

# Make sure that names are the same across country level datasets to prepare for merging
names(UKAgg) <- names(MexAgg)
names(SpainAgg) <- names(MexAgg)
names(PortAgg) <- names(MexAgg)
names(BrazilAgg) <- names(MexAgg)
names(ArgAgg) <- names(MexAgg)
names(NZAgg) <- names(MexAgg)
names(NorAusFranceAgg) <- names(MexAgg)

# Merge (rbind) all datasets
MergedN <- rbind(MexAgg, PeruAgg, UKAgg, SpainAgg, PortAgg, USAgg, BrazilAgg, ArgAgg, NZAgg, NorAusFranceAgg)

cat("Merged dataset has", nrow(MergedN), "rows and", ncol(MergedN), "columns\n")

# Ensure treatment numbers are the same across datasets
MergedN$treat <- recode(MergedN$treat, "'treat1' = 1; 'treat2' = 2; 'treat3' = 3; 'treat4' = 4; 'treat5' = 5; 'treat6' = 6")

# Recode gender variables for consistency across datasets
MergedN[MergedN == "Female"] <- 1
MergedN[MergedN == "Feminino"] <- 1
MergedN[MergedN == "Mujer"] <- 1
MergedN[MergedN == "Woman"] <- 1
MergedN[MergedN == "Hombre"] <- 0
MergedN[MergedN == "Male"] <- 0
MergedN[MergedN == "Man"] <- 0
MergedN[MergedN == "Masculino"] <- 0
MergedN[MergedN == "Other"] <- NA
MergedN[MergedN == "Otro/ No binario"] <- NA

# Recode values on political ideology, for consistency across datasets
MergedN[MergedN == "Left\n(0)"] <- 0
MergedN[MergedN == "Right\n(10)"] <- 10

MergedN$leftright_1 <- as.numeric(as.character(MergedN$leftright_1))
MergedN$gender <- as.numeric(as.character(MergedN$gender))

# Subset dataset for those who saw a vignette about sexual harassment
harass <- subset(MergedN, treat == 1 | treat == 2 | treat == 3)
cat("Sexual harassment subset:", nrow(harass), "rows\n")

# Subset dataset for those who saw animal mistreatment vignette
animal <- subset(MergedN, treat == 4 | treat == 5 | treat == 6)
cat("Animal mistreatment subset:", nrow(animal), "rows\n")

cat("Data loading and merging complete.\n")


## 2. Merged Cross-Country Analysis


### 2. Merged Cross-Country Analysis

Combine country datasets and generate main figures:

**Figure 2:** Main treatment effects (sexual harassment vignettes)
- Substantive legitimacy by council composition
- Procedural legitimacy by council composition
- Pooled 12-country sample with 95% confidence intervals

**Figure 4:** USA vs UK comparison
- Treatment effects separately for two non-quota countries
- Highlights quota penalty in countries without statutory quotas

Treatment conditions:
1. All-male council (8 men)
2. Gender-balanced council (4 men, 4 women)
3. Quota-elected gender-balanced council (4 men, 4 women via quota rule)

In [ ]:
# Generate all figures

cat("Generating figures...\n")

# Position dodge for error bars
pd <- position_dodge(0.1)

### FIGURE 2: Women's rights issue (sexual harassment) - Substantive and Procedural legitimacy ###

cat("Creating Figure 2...\n")

# Remove NA values from substantive legitimacy DV
harass_fig2 <- harass[!is.na(harass$SubLegStand), ]

# Get averages and CIs for substantive legitimacy across treatments
data_sum1 <- summarySE(harass_fig2, measurevar = "SubLegStand", groupvars = c("treat"))

# Plot substantive legitimacy
plot1 <- ggplot(data_sum1, aes(x = 1, y = SubLegStand, group = treat)) +
  geom_errorbar(aes(ymin = SubLegStand - se, ymax = SubLegStand + se), colour = "black", width = .1, position = pd) +
  geom_line(position = pd) +
  geom_point(position = pd, size = 4, shape = 21, fill = "white") +
  xlab("") +
  ylab("Average Substantive Legitimacy \n (standardized within countries)") +
  scale_y_continuous(limits = c(-1, 1)) +
  ggtitle("Women's rights issue (sexual harassment) \n Substantive legitimacy beliefs") +
  theme(axis.title.x = element_blank(),
        axis.text.x = element_blank(),
        axis.ticks.x = element_blank(),
        axis.text.y = element_text(size = 12)) +
  theme(plot.title = element_text(size = 12))

# Remove NA values from procedural legitimacy DV
harass_fig2 <- harass_fig2[!is.na(harass_fig2$ProLeg), ]

# Get averages and CIs for procedural legitimacy across treatments
data_sum2 <- summarySE(harass_fig2, measurevar = "ProLegStand", groupvars = c("treat"))

# Plot procedural legitimacy
plot2 <- ggplot(data_sum2, aes(x = 1, y = ProLegStand, group = treat)) +
  geom_errorbar(aes(ymin = ProLegStand - se, ymax = ProLegStand + se), colour = "black", width = .1, position = pd) +
  geom_line(position = pd) +
  geom_point(position = pd, size = 4, shape = 21, fill = "white") +
  xlab("") +
  ylab("Average Procedural Legitimacy \n (standardized within countries)") +
  scale_y_continuous(limits = c(-1, 1)) +
  ggtitle("Women's rights issue (sexual harassment) \n Procedural legitimacy beliefs") +
  theme(axis.title.x = element_blank(),
        axis.text.x = element_blank(),
        axis.ticks.x = element_blank(),
        axis.text.y = element_text(size = 12)) +
  theme(plot.title = element_text(size = 12))

# Save Figure 2
pdf(file.path(figures_dir, "Figure2.pdf"), width = 8.2, height = 5)
grid.arrange(plot1 + annotate("text", x = 0.9675, y = -0.05, label = "All-Male Council", size = 3.3) +
               annotate("text", x = 1, y = 0.28, label = "Gender-Balanced \n Council", size = 3.3) +
               annotate("text", x = 1.032, y = 0.25, label = "Quota-Elected \n Gender-Balanced \n Council", size = 3.3),
             plot2 + annotate("text", x = 0.9675, y = -0.42, label = "All-Male Council", size = 3.3) +
               annotate("text", x = 1, y = 0.45, label = "Gender-Balanced \n Council", size = 3.3) +
               annotate("text", x = 1.032, y = 0.38, label = "Quota-Elected \n Gender-Balanced \n Council", size = 3.3),
             ncol = 2, nrow = 1)
dev.off()

cat("Figure 2 saved.\n")

### FIGURE 4: USA vs. UK ###

cat("Creating Figure 4...\n")

# Subset data for UK and US
USA <- subset(harass, Country == "USA")
UK <- subset(harass, Country == "UK")

# USA plots
data_sum1 <- summarySE(USA, measurevar = "SubLegStand", groupvars = c("treat"))
plot1 <- ggplot(data_sum1, aes(x = 1, y = SubLegStand, group = treat)) +
  geom_errorbar(aes(ymin = SubLegStand - se, ymax = SubLegStand + se), colour = "black", width = .1, position = pd) +
  geom_line(position = pd) +
  geom_point(position = pd, size = 3, shape = 21, fill = "white") +
  xlab("") +
  ylab("Average Substantive Legitimacy \n (standardized within countries)") +
  scale_y_continuous(limits = c(-1, 1)) +
  ggtitle("Women's rights issue (sexual harassment) \n Substantive legitimacy beliefs \n US sample") +
  theme(axis.title.x = element_blank(),
        axis.text.x = element_blank(),
        axis.ticks.x = element_blank(),
        axis.text.y = element_text(size = 12)) +
  theme(plot.title = element_text(size = 12))

data_sum2 <- summarySE(USA, measurevar = "ProLegStand", groupvars = c("treat"))
plot2 <- ggplot(data_sum2, aes(x = 1, y = ProLegStand, group = treat)) +
  geom_errorbar(aes(ymin = ProLegStand - se, ymax = ProLegStand + se), colour = "black", width = .1, position = pd) +
  geom_line(position = pd) +
  geom_point(position = pd, size = 3, shape = 21, fill = "white") +
  xlab("") +
  ylab("Average Procedural Legitimacy \n (standardized within countries)") +
  scale_y_continuous(limits = c(-1, 1)) +
  ggtitle("Women's rights issue (sexual harassment) \n Procedural legitimacy beliefs \n US sample") +
  theme(axis.title.x = element_blank(),
        axis.text.x = element_blank(),
        axis.ticks.x = element_blank(),
        axis.text.y = element_text(size = 12)) +
  theme(plot.title = element_text(size = 12))

# UK plots
data_sum1 <- summarySE(UK, measurevar = "SubLegStand", groupvars = c("treat"))
plot3 <- ggplot(data_sum1, aes(x = 1, y = SubLegStand, group = treat)) +
  geom_errorbar(aes(ymin = SubLegStand - se, ymax = SubLegStand + se), colour = "black", width = .1, position = pd) +
  geom_line(position = pd) +
  geom_point(position = pd, size = 3, shape = 21, fill = "white") +
  xlab("") +
  ylab("Average Substantive Legitimacy \n (standardized within country)") +
  scale_y_continuous(limits = c(-1, 1)) +
  ggtitle("Women's rights issue (sexual harassment) \n Substantive legitimacy beliefs \n UK sample") +
  theme(axis.title.x = element_blank(),
        axis.text.x = element_blank(),
        axis.ticks.x = element_blank(),
        axis.text.y = element_text(size = 12)) +
  theme(plot.title = element_text(size = 12))

data_sum2 <- summarySE(UK, measurevar = "ProLegStand", groupvars = c("treat"))
plot4 <- ggplot(data_sum2, aes(x = 1, y = ProLegStand, group = treat)) +
  geom_errorbar(aes(ymin = ProLegStand - se, ymax = ProLegStand + se), colour = "black", width = .1, position = pd) +
  geom_line(position = pd) +
  geom_point(position = pd, size = 3, shape = 21, fill = "white") +
  xlab("") +
  ylab("Average Procedural Legitimacy \n (standardized within country)") +
  scale_y_continuous(limits = c(-1, 1)) +
  ggtitle("Women's rights issue (sexual harassment) \n Procedural legitimacy beliefs \n UK sample") +
  theme(axis.title.x = element_blank(),
        axis.text.x = element_blank(),
        axis.ticks.x = element_blank(),
        axis.text.y = element_text(size = 12)) +
  theme(plot.title = element_text(size = 12))

# Save Figure 4
pdf(file.path(figures_dir, "Figure4.pdf"), width = 8.6, height = 5.6)
grid.arrange(plot1 + annotate("text", x = 0.9675, y = 0.15, label = "All-Male Council", size = 3.3) +
               annotate("text", x = 1, y = 0.45, label = "Gender-Balanced \n Council", size = 3.3) +
               annotate("text", x = 1.032, y = 0.25, label = "Quota-Elected \n Gender-Balanced \n Council", size = 3.3),
             plot2 + annotate("text", x = 0.9675, y = -0.05, label = "All-Male Council", size = 3.3) +
               annotate("text", x = 1, y = 0.55, label = "Gender-Balanced \n Council", size = 3.3) +
               annotate("text", x = 1.032, y = 0.42, label = "Quota-Elected \n Gender-Balanced \n Council", size = 3.3),
             plot3 + annotate("text", x = 0.9675, y = 0.15, label = "All-Male Council", size = 3.3) +
               annotate("text", x = 1, y = 0.45, label = "Gender-Balanced \n Council", size = 3.3) +
               annotate("text", x = 1.032, y = 0.25, label = "Quota-Elected \n Gender-Balanced \n Council", size = 3.3),
             plot4 + annotate("text", x = 0.9675, y = -0.15, label = "All-Male Council", size = 3.3) +
               annotate("text", x = 1, y = 0.65, label = "Gender-Balanced \n Council", size = 3.3) +
               annotate("text", x = 1.032, y = 0.42, label = "Quota-Elected \n Gender-Balanced \n Council", size = 3.3),
             ncol = 2, nrow = 2)
dev.off()

cat("Figure 4 saved.\n")

### FIGURE 5: Non-women's rights issue (animal mistreatment) ###

cat("Creating Figure 5...\n")

# Remove NA values from DVs
animal <- animal[!is.na(animal$SubLegStand), ]

# Get averages and CIs for substantive legitimacy across treatments
data_sum1 <- summarySE(animal, measurevar = "SubLegStand", groupvars = c("treat"))

plot3 <- ggplot(data_sum1, aes(x = 1, y = SubLegStand, group = treat)) +
  geom_errorbar(aes(ymin = SubLegStand - se, ymax = SubLegStand + se), colour = "black", width = .1, position = pd) +
  geom_line(position = pd) +
  geom_point(position = pd, size = 4, shape = 21, fill = "white") +
  xlab("") +
  ylab("Average Substantive Legitimacy \n (standardized within countries)") +
  scale_y_continuous(limits = c(-1, 1)) +
  ggtitle("Non women's rights issue (animal mistreatment) \n Substantive legitimacy beliefs") +
  theme(axis.title.x = element_blank(),
        axis.text.x = element_blank(),
        axis.ticks.x = element_blank(),
        axis.text.y = element_text(size = 5)) +
  theme(plot.title = element_text(size = 11))

animal <- animal[!is.na(animal$ProLeg), ]

data_sum2 <- summarySE(animal, measurevar = "ProLegStand", groupvars = c("treat"))

plot4 <- ggplot(data_sum2, aes(x = 1, y = ProLegStand, group = treat)) +
  geom_errorbar(aes(ymin = ProLegStand - se, ymax = ProLegStand + se), colour = "black", width = .1, position = pd) +
  geom_line(position = pd) +
  geom_point(position = pd, size = 4, shape = 21, fill = "white") +
  xlab("") +
  ylab("Average Procedural Legitimacy \n (standardized within countries)") +
  scale_y_continuous(limits = c(-1, 1)) +
  ggtitle("Non women's rights issue (animal mistreatment) \n Procedural legitimacy beliefs") +
  theme(axis.title.x = element_blank(),
        axis.text.x = element_blank(),
        axis.ticks.x = element_blank(),
        axis.text.y = element_text(size = 5)) +
  theme(plot.title = element_text(size = 11))

# Save Figure 5
pdf(file.path(figures_dir, "Figure5.pdf"), width = 8.6, height = 5.6)
grid.arrange(plot3 + annotate("text", x = 0.9675, y = 0.03, label = "All-Male Council", size = 3.3) +
               annotate("text", x = 1, y = 0.2, label = "Gender-Balanced \n Council", size = 3.3) +
               annotate("text", x = 1.032, y = 0.2, label = "Quota-Elected \n Gender-Balanced \n Council", size = 3.3),
             plot4 + annotate("text", x = 0.9675, y = -0.36, label = "All-Male Council", size = 3.3) +
               annotate("text", x = 1, y = 0.4, label = "Gender-Balanced \n Council", size = 3.3) +
               annotate("text", x = 1.032, y = 0.35, label = "Quota-Elected \n Gender-Balanced \n Council", size = 3.3),
             ncol = 2, nrow = 1)
dev.off()

cat("Figure 5 saved.\n")

cat("All figures generated successfully.\n")


## 3. Balance Tests (Randomization Check)


### 3. Balance Tests (Randomization Check)

Verify experimental randomization by comparing treatment groups on
pre-treatment covariates.

**Table SI.4:** Sexual harassment treatments (treats 1-3)
- Gender distribution across treatments
- Ideology (left-right scale) across treatments
- Country distribution across treatments

**Table SI.5:** Animal mistreatment treatments (treats 4-6)
- Same balance checks for non-gendered issue vignettes

Expected result: No significant differences (randomization successful)

In [ ]:
# Generate all supplementary tables

cat("Generating tables...\n")

# Reload harass and animal from merged data (in case modified by figures script)
harass <- subset(MergedN, treat == 1 | treat == 2 | treat == 3)
animal <- subset(MergedN, treat == 4 | treat == 5 | treat == 6)

### TABLE SI.4: Balance on women's issue treatments ###

cat("Creating Table SI.4...\n")

# Create country dummy variables
harass$Argentina <- recode(harass$Country, "'Argentina' = 1; else = 0")
harass$Australia <- recode(harass$Country, "'Australia' = 1; else = 0")
harass$Brazil <- recode(harass$Country, "'Brazil' = 1; else = 0")
harass$France <- recode(harass$Country, "'France' = 1; else = 0")
harass$Mexico <- recode(harass$Country, "'Mexico' = 1; else = 0")
harass$Norway <- recode(harass$Country, "'Norway' = 1; else = 0")
harass$NZ <- recode(harass$Country, "'NZ' = 1; else = 0")
harass$Peru <- recode(harass$Country, "'Peru' = 1; else = 0")
harass$Portugal <- recode(harass$Country, "'Portugal' = 1; else = 0")
harass$Spain <- recode(harass$Country, "'Spain' = 1; else = 0")
harass$UK <- recode(harass$Country, "'UK' = 1; else = 0")
harass$USA <- recode(harass$Country, "'USA' = 1; else = 0")

# Calculate means by treatment
tgender <- tapply(harass$gender, harass$treat, mean, na.rm = TRUE)
tideo <- tapply(harass$leftright_1, harass$treat, mean, na.rm = TRUE)
targ <- tapply(harass$Argentina, harass$treat, mean, na.rm = TRUE)
taus <- tapply(harass$Australia, harass$treat, mean, na.rm = TRUE)
tbra <- tapply(harass$Brazil, harass$treat, mean, na.rm = TRUE)
tfra <- tapply(harass$France, harass$treat, mean, na.rm = TRUE)
tmex <- tapply(harass$Mexico, harass$treat, mean, na.rm = TRUE)
tnor <- tapply(harass$Norway, harass$treat, mean, na.rm = TRUE)
tnz <- tapply(harass$NZ, harass$treat, mean, na.rm = TRUE)
tper <- tapply(harass$Peru, harass$treat, mean, na.rm = TRUE)
tpor <- tapply(harass$Portugal, harass$treat, mean, na.rm = TRUE)
tspa <- tapply(harass$Spain, harass$treat, mean, na.rm = TRUE)
tuk <- tapply(harass$UK, harass$treat, mean, na.rm = TRUE)
tusa <- tapply(harass$USA, harass$treat, mean, na.rm = TRUE)

tapply_df <- as.data.frame(rbind(tgender, tideo, targ, taus, tbra, tfra, tmex, tnor, tnz, tper, tpor, tspa, tuk, tusa))
latex_table <- xtable(tapply_df)

sink(file.path(tables_dir, "tableSI.4.tex"))
print(latex_table, type = "latex")
sink()

cat("Table SI.4 saved.\n")

### TABLE SI.5: Balance on animal treatment treatments ###

cat("Creating Table SI.5...\n")

# Create country dummy variables for animal
animal$Argentina <- recode(animal$Country, "'Argentina' = 1; else = 0")
animal$Australia <- recode(animal$Country, "'Australia' = 1; else = 0")
animal$Brazil <- recode(animal$Country, "'Brazil' = 1; else = 0")
animal$France <- recode(animal$Country, "'France' = 1; else = 0")
animal$Mexico <- recode(animal$Country, "'Mexico' = 1; else = 0")
animal$Norway <- recode(animal$Country, "'Norway' = 1; else = 0")
animal$NZ <- recode(animal$Country, "'NZ' = 1; else = 0")
animal$Peru <- recode(animal$Country, "'Peru' = 1; else = 0")
animal$Portugal <- recode(animal$Country, "'Portugal' = 1; else = 0")
animal$Spain <- recode(animal$Country, "'Spain' = 1; else = 0")
animal$UK <- recode(animal$Country, "'UK' = 1; else = 0")
animal$USA <- recode(animal$Country, "'USA' = 1; else = 0")

tgender <- tapply(animal$gender, animal$treat, mean, na.rm = TRUE)
tideo <- tapply(animal$leftright_1, animal$treat, mean, na.rm = TRUE)
targ <- tapply(animal$Argentina, animal$treat, mean, na.rm = TRUE)
taus <- tapply(animal$Australia, animal$treat, mean, na.rm = TRUE)
tbra <- tapply(animal$Brazil, animal$treat, mean, na.rm = TRUE)
tfra <- tapply(animal$France, animal$treat, mean, na.rm = TRUE)
tmex <- tapply(animal$Mexico, animal$treat, mean, na.rm = TRUE)
tnor <- tapply(animal$Norway, animal$treat, mean, na.rm = TRUE)
tnz <- tapply(animal$NZ, animal$treat, mean, na.rm = TRUE)
tper <- tapply(animal$Peru, animal$treat, mean, na.rm = TRUE)
tpor <- tapply(animal$Portugal, animal$treat, mean, na.rm = TRUE)
tspa <- tapply(animal$Spain, animal$treat, mean, na.rm = TRUE)
tuk <- tapply(animal$UK, animal$treat, mean, na.rm = TRUE)
tusa <- tapply(animal$USA, animal$treat, mean, na.rm = TRUE)

tapply_df <- as.data.frame(rbind(tgender, tideo, targ, taus, tbra, tfra, tmex, tnor, tnz, tper, tpor, tspa, tuk, tusa))
latex_table <- xtable(tapply_df)

sink(file.path(tables_dir, "tableSI.5.tex"))
print(latex_table, type = "latex")
sink()

cat("Table SI.5 saved.\n")

### TABLE SI.6: Main cross-country treatment effects ###

cat("Creating Table SI.6...\n")

# Perform t-tests
t1 <- t.test(SubLegStand ~ treat, data = harass[harass$treat %in% c(1, 2), ])
t2 <- t.test(SubLegStand ~ treat, data = harass[harass$treat %in% c(1, 3), ])
t3 <- t.test(ProLegStand ~ treat, data = harass[harass$treat %in% c(1, 2), ])
t4 <- t.test(ProLegStand ~ treat, data = harass[harass$treat %in% c(1, 3), ])

# Create results data frame
results <- data.frame(
  Variable = c("SubLegStand", "SubLegStand", "ProLegStand", "ProLegStand"),
  Comparison = c("All Male vs. Gender-Balanced",
                 "All Male vs. Quota Gender-Balanced",
                 "All Male vs. Gender-Balanced",
                 "All Male vs. Quota Gender-Balanced"),
  t_stat = c(t1$statistic, t2$statistic, t3$statistic, t4$statistic),
  p_value = c(t1$p.value, t2$p.value, t3$p.value, t4$p.value),
  CI_lower = c(t1$conf.int[1], t2$conf.int[1], t3$conf.int[1], t4$conf.int[1]),
  CI_upper = c(t1$conf.int[2], t2$conf.int[2], t3$conf.int[2], t4$conf.int[2])
)

rownames(results) <- c("T1: SubLegStand (1 vs. 2)",
                       "T2: SubLegStand (1 vs. 3)",
                       "T3: ProLegStand (1 vs. 2)",
                       "T4: ProLegStand (1 vs. 3)")

latex_table <- xtable(results)

sink(file.path(tables_dir, "tableSI.6.tex"))
print(latex_table, type = "latex")
sink()

cat("Table SI.6 saved.\n")

### TABLE SI.8: Treatment effects for US and UK ###

cat("Creating Table SI.8...\n")

USA <- subset(harass, Country == "USA")
UK <- subset(harass, Country == "UK")

# Perform t-tests for the US
t1 <- t.test(SubLegStand ~ treat, data = USA[USA$treat %in% c(1, 2), ])
t2 <- t.test(SubLegStand ~ treat, data = USA[USA$treat %in% c(1, 3), ])
t3 <- t.test(ProLegStand ~ treat, data = USA[USA$treat %in% c(1, 2), ])
t4 <- t.test(ProLegStand ~ treat, data = USA[USA$treat %in% c(1, 3), ])

# Perform t-tests for the UK
t5 <- t.test(SubLegStand ~ treat, data = UK[UK$treat %in% c(1, 2), ])
t6 <- t.test(SubLegStand ~ treat, data = UK[UK$treat %in% c(1, 3), ])
t7 <- t.test(ProLegStand ~ treat, data = UK[UK$treat %in% c(1, 2), ])
t8 <- t.test(ProLegStand ~ treat, data = UK[UK$treat %in% c(1, 3), ])

# Create results data frame
results <- data.frame(
  Country = c(rep("USA", 4), rep("UK", 4)),
  Variable = c(rep("SubLegStand", 2), rep("ProLegStand", 2),
               rep("SubLegStand", 2), rep("ProLegStand", 2)),
  Comparison = c("All Male vs. Gender-Balanced", "All Male vs. Quota Gender-Balanced",
                 "All Male vs. Gender-Balanced", "All Male vs. Quota Gender-Balanced",
                 "All Male vs. Gender-Balanced", "All Male vs. Quota Gender-Balanced",
                 "All Male vs. Gender-Balanced", "All Male vs. Quota Gender-Balanced"),
  t_stat = c(t1$statistic, t2$statistic, t3$statistic, t4$statistic,
             t5$statistic, t6$statistic, t7$statistic, t8$statistic),
  p_value = c(t1$p.value, t2$p.value, t3$p.value, t4$p.value,
              t5$p.value, t6$p.value, t7$p.value, t8$p.value),
  CI_lower = c(t1$conf.int[1], t2$conf.int[1], t3$conf.int[1], t4$conf.int[1],
               t5$conf.int[1], t6$conf.int[1], t7$conf.int[1], t8$conf.int[1]),
  CI_upper = c(t1$conf.int[2], t2$conf.int[2], t3$conf.int[2], t4$conf.int[2],
               t5$conf.int[2], t6$conf.int[2], t7$conf.int[2], t8$conf.int[2])
)

rownames(results) <- c("SI.8: USA - T1", "SI.8: USA - T2", "SI.8: USA - T3", "SI.8: USA - T4",
                       "SI.8: UK - T5", "SI.8: UK - T6", "SI.8: UK - T7", "SI.8: UK - T8")

latex_table <- xtable(results)

sink(file.path(tables_dir, "tableSI.8.tex"))
print(latex_table, type = "latex")
sink()

cat("Table SI.8 saved.\n")

### TABLE SI.9: Treatment effects for animal mistreatment ###

cat("Creating Table SI.9...\n")

# Perform t-tests for animal mistreatment vignettes
t1 <- t.test(SubLegStand ~ treat, data = animal[animal$treat %in% c(4, 5), ])
t2 <- t.test(SubLegStand ~ treat, data = animal[animal$treat %in% c(4, 6), ])
t3 <- t.test(ProLegStand ~ treat, data = animal[animal$treat %in% c(4, 5), ])
t4 <- t.test(ProLegStand ~ treat, data = animal[animal$treat %in% c(4, 6), ])

results <- data.frame(
  Variable = c(rep("SubLegStand", 2), rep("ProLegStand", 2)),
  Comparison = c("All Male vs. Gender-Balanced", "All Male vs. Quota Gender-Balanced",
                 "All Male vs. Gender-Balanced", "All Male vs. Quota Gender-Balanced"),
  t_stat = c(t1$statistic, t2$statistic, t3$statistic, t4$statistic),
  p_value = c(t1$p.value, t2$p.value, t3$p.value, t4$p.value),
  CI_lower = c(t1$conf.int[1], t2$conf.int[1], t3$conf.int[1], t4$conf.int[1]),
  CI_upper = c(t1$conf.int[2], t2$conf.int[2], t3$conf.int[2], t4$conf.int[2])
)

rownames(results) <- c("SI.9: Animal - T1", "SI.9: Animal - T2", "SI.9: Animal - T3", "SI.9: Animal - T4")

latex_table <- xtable(results)

sink(file.path(tables_dir, "tableSI9.tex"))
print(latex_table, type = "latex")
sink()

cat("Table SI.9 saved.\n")

### TABLE SI.10: Model-based specification with covariates - substantive legitimacy ###

cat("Creating Table SI.10...\n")

harassGB <- subset(harass, treat == 2 | treat == 3)
harassAM_GB <- subset(harass, treat == 1 | treat == 2)
harassAM_GBQ <- subset(harass, treat == 1 | treat == 3)

harassAM_GBQ$treat <- recode(harassAM_GBQ$treat, "3 = 2")

# Main models for outcome of substantive legitimacy
model1 <- lm(SubLegStand ~ treat + gender + leftright_1 + Country, data = harassAM_GB)
model2 <- lm(SubLegStand ~ treat + gender + leftright_1 + Country, data = harassAM_GBQ)
model3 <- lm(SubLegStand ~ treat + gender + leftright_1 + Country, data = harassGB)

# Extract coefficients
extract_coefficients <- function(model) {
  summary_model <- summary(model)
  coefs <- summary_model$coefficients[, 1:2]
  return(data.frame(Estimate = coefs[, 1], StdError = coefs[, 2]))
}

models <- list(model1, model2, model3)
model_results <- lapply(models, extract_coefficients)

coef_names <- rownames(model_results[[1]])
results_df <- data.frame(
  Coefficient = coef_names,
  Model1_Estimate = model_results[[1]]$Estimate,
  Model1_StdError = model_results[[1]]$StdError,
  Model2_Estimate = model_results[[2]]$Estimate,
  Model2_StdError = model_results[[2]]$StdError,
  Model3_Estimate = model_results[[3]]$Estimate,
  Model3_StdError = model_results[[3]]$StdError
)

latex_table <- xtable(results_df)

sink(file.path(tables_dir, "tableSI10.tex"))
print(latex_table, type = "latex")
sink()

cat("Table SI.10 saved.\n")

### TABLE SI.11: Model-based specification with covariates - procedural legitimacy ###

cat("Creating Table SI.11...\n")

model1 <- lm(ProLegStand ~ treat + gender + leftright_1 + Country, data = harassAM_GB)
model2 <- lm(ProLegStand ~ treat + gender + leftright_1 + Country, data = harassAM_GBQ)
model3 <- lm(ProLegStand ~ treat + gender + leftright_1 + Country, data = harassGB)

models <- list(model1, model2, model3)
model_results <- lapply(models, extract_coefficients)

coef_names <- rownames(model_results[[1]])
results_df <- data.frame(
  Coefficient = coef_names,
  Model1_Estimate = model_results[[1]]$Estimate,
  Model1_StdError = model_results[[1]]$StdError,
  Model2_Estimate = model_results[[2]]$Estimate,
  Model2_StdError = model_results[[2]]$StdError,
  Model3_Estimate = model_results[[3]]$Estimate,
  Model3_StdError = model_results[[3]]$StdError
)

latex_table <- xtable(results_df)

sink(file.path(tables_dir, "tableSI11.tex"))
print(latex_table, type = "latex")
sink()

cat("Table SI.11 saved.\n")

### TABLE SI.12: Robustness check excluding political left - substantive legitimacy ###

cat("Creating Table SI.12...\n")

# Excluding political liberals -- those that score five or below on the ideological self placement scale
harassAM_GB_ModCon <- subset(harassAM_GB, leftright_1 > 4)
harassAM_GBQ_ModCon <- subset(harassAM_GBQ, leftright_1 > 4)
harassGB_ModCon <- subset(harassGB, leftright_1 > 4)

model1 <- lm(SubLegStand ~ treat + gender + leftright_1 + Country, data = harassAM_GB_ModCon)
model2 <- lm(SubLegStand ~ treat + gender + leftright_1 + Country, data = harassAM_GBQ_ModCon)
model3 <- lm(SubLegStand ~ treat + gender + leftright_1 + Country, data = harassGB_ModCon)

models <- list(model1, model2, model3)
model_results <- lapply(models, extract_coefficients)

coef_names <- rownames(model_results[[1]])
results_df <- data.frame(
  Coefficient = coef_names,
  Model1_Estimate = model_results[[1]]$Estimate,
  Model1_StdError = model_results[[1]]$StdError,
  Model2_Estimate = model_results[[2]]$Estimate,
  Model2_StdError = model_results[[2]]$StdError,
  Model3_Estimate = model_results[[3]]$Estimate,
  Model3_StdError = model_results[[3]]$StdError
)

latex_table <- xtable(results_df)

sink(file.path(tables_dir, "tableSI12.tex"))
print(latex_table, type = "latex")
sink()

cat("Table SI.12 saved.\n")

### TABLE SI.13: Robustness check excluding political left - procedural legitimacy ###

cat("Creating Table SI.13...\n")

model1 <- lm(ProLegStand ~ treat + gender + leftright_1 + Country, data = harassAM_GB_ModCon)
model2 <- lm(ProLegStand ~ treat + gender + leftright_1 + Country, data = harassAM_GBQ_ModCon)
model3 <- lm(ProLegStand ~ treat + gender + leftright_1 + Country, data = harassGB_ModCon)

models <- list(model1, model2, model3)
model_results <- lapply(models, extract_coefficients)

coef_names <- rownames(model_results[[1]])
results_df <- data.frame(
  Coefficient = coef_names,
  Model1_Estimate = model_results[[1]]$Estimate,
  Model1_StdError = model_results[[1]]$StdError,
  Model2_Estimate = model_results[[2]]$Estimate,
  Model2_StdError = model_results[[2]]$StdError,
  Model3_Estimate = model_results[[3]]$Estimate,
  Model3_StdError = model_results[[3]]$StdError
)

latex_table <- xtable(results_df)

sink(file.path(tables_dir, "tableSI13.tex"))
print(latex_table, type = "latex")
sink()

cat("Table SI.13 saved.\n")

cat("All tables generated successfully.\n")


## 4. Main Treatment Effects (T-Tests)


### 4. Main Treatment Effects (T-Tests)

Bivariate comparisons of legitimacy outcomes across treatment conditions.

**Table SI.6:** Sexual harassment vignettes (12 countries pooled)
- All-Male vs Gender-Balanced (treats 1 vs 2)
- All-Male vs Quota Gender-Balanced (treats 1 vs 3)
- Gender-Balanced vs Quota Gender-Balanced (treats 2 vs 3) = "quota penalty"
- For both substantive and procedural legitimacy

**Table SI.8:** USA and UK specific effects
- Country-specific treatment effects for two non-quota countries

**Table SI.9:** Animal mistreatment vignettes (9 countries)
- Same treatment comparisons for non-gendered issue

Method: Two-sample t-tests with 95% confidence intervals

In [ ]:
# Generate all supplementary tables

cat("Generating tables...\n")

# Reload harass and animal from merged data (in case modified by figures script)
harass <- subset(MergedN, treat == 1 | treat == 2 | treat == 3)
animal <- subset(MergedN, treat == 4 | treat == 5 | treat == 6)

### TABLE SI.4: Balance on women's issue treatments ###

cat("Creating Table SI.4...\n")

# Create country dummy variables
harass$Argentina <- recode(harass$Country, "'Argentina' = 1; else = 0")
harass$Australia <- recode(harass$Country, "'Australia' = 1; else = 0")
harass$Brazil <- recode(harass$Country, "'Brazil' = 1; else = 0")
harass$France <- recode(harass$Country, "'France' = 1; else = 0")
harass$Mexico <- recode(harass$Country, "'Mexico' = 1; else = 0")
harass$Norway <- recode(harass$Country, "'Norway' = 1; else = 0")
harass$NZ <- recode(harass$Country, "'NZ' = 1; else = 0")
harass$Peru <- recode(harass$Country, "'Peru' = 1; else = 0")
harass$Portugal <- recode(harass$Country, "'Portugal' = 1; else = 0")
harass$Spain <- recode(harass$Country, "'Spain' = 1; else = 0")
harass$UK <- recode(harass$Country, "'UK' = 1; else = 0")
harass$USA <- recode(harass$Country, "'USA' = 1; else = 0")

# Calculate means by treatment
tgender <- tapply(harass$gender, harass$treat, mean, na.rm = TRUE)
tideo <- tapply(harass$leftright_1, harass$treat, mean, na.rm = TRUE)
targ <- tapply(harass$Argentina, harass$treat, mean, na.rm = TRUE)
taus <- tapply(harass$Australia, harass$treat, mean, na.rm = TRUE)
tbra <- tapply(harass$Brazil, harass$treat, mean, na.rm = TRUE)
tfra <- tapply(harass$France, harass$treat, mean, na.rm = TRUE)
tmex <- tapply(harass$Mexico, harass$treat, mean, na.rm = TRUE)
tnor <- tapply(harass$Norway, harass$treat, mean, na.rm = TRUE)
tnz <- tapply(harass$NZ, harass$treat, mean, na.rm = TRUE)
tper <- tapply(harass$Peru, harass$treat, mean, na.rm = TRUE)
tpor <- tapply(harass$Portugal, harass$treat, mean, na.rm = TRUE)
tspa <- tapply(harass$Spain, harass$treat, mean, na.rm = TRUE)
tuk <- tapply(harass$UK, harass$treat, mean, na.rm = TRUE)
tusa <- tapply(harass$USA, harass$treat, mean, na.rm = TRUE)

tapply_df <- as.data.frame(rbind(tgender, tideo, targ, taus, tbra, tfra, tmex, tnor, tnz, tper, tpor, tspa, tuk, tusa))
latex_table <- xtable(tapply_df)

sink(file.path(tables_dir, "tableSI.4.tex"))
print(latex_table, type = "latex")
sink()

cat("Table SI.4 saved.\n")

### TABLE SI.5: Balance on animal treatment treatments ###

cat("Creating Table SI.5...\n")

# Create country dummy variables for animal
animal$Argentina <- recode(animal$Country, "'Argentina' = 1; else = 0")
animal$Australia <- recode(animal$Country, "'Australia' = 1; else = 0")
animal$Brazil <- recode(animal$Country, "'Brazil' = 1; else = 0")
animal$France <- recode(animal$Country, "'France' = 1; else = 0")
animal$Mexico <- recode(animal$Country, "'Mexico' = 1; else = 0")
animal$Norway <- recode(animal$Country, "'Norway' = 1; else = 0")
animal$NZ <- recode(animal$Country, "'NZ' = 1; else = 0")
animal$Peru <- recode(animal$Country, "'Peru' = 1; else = 0")
animal$Portugal <- recode(animal$Country, "'Portugal' = 1; else = 0")
animal$Spain <- recode(animal$Country, "'Spain' = 1; else = 0")
animal$UK <- recode(animal$Country, "'UK' = 1; else = 0")
animal$USA <- recode(animal$Country, "'USA' = 1; else = 0")

tgender <- tapply(animal$gender, animal$treat, mean, na.rm = TRUE)
tideo <- tapply(animal$leftright_1, animal$treat, mean, na.rm = TRUE)
targ <- tapply(animal$Argentina, animal$treat, mean, na.rm = TRUE)
taus <- tapply(animal$Australia, animal$treat, mean, na.rm = TRUE)
tbra <- tapply(animal$Brazil, animal$treat, mean, na.rm = TRUE)
tfra <- tapply(animal$France, animal$treat, mean, na.rm = TRUE)
tmex <- tapply(animal$Mexico, animal$treat, mean, na.rm = TRUE)
tnor <- tapply(animal$Norway, animal$treat, mean, na.rm = TRUE)
tnz <- tapply(animal$NZ, animal$treat, mean, na.rm = TRUE)
tper <- tapply(animal$Peru, animal$treat, mean, na.rm = TRUE)
tpor <- tapply(animal$Portugal, animal$treat, mean, na.rm = TRUE)
tspa <- tapply(animal$Spain, animal$treat, mean, na.rm = TRUE)
tuk <- tapply(animal$UK, animal$treat, mean, na.rm = TRUE)
tusa <- tapply(animal$USA, animal$treat, mean, na.rm = TRUE)

tapply_df <- as.data.frame(rbind(tgender, tideo, targ, taus, tbra, tfra, tmex, tnor, tnz, tper, tpor, tspa, tuk, tusa))
latex_table <- xtable(tapply_df)

sink(file.path(tables_dir, "tableSI.5.tex"))
print(latex_table, type = "latex")
sink()

cat("Table SI.5 saved.\n")

### TABLE SI.6: Main cross-country treatment effects ###

cat("Creating Table SI.6...\n")

# Perform t-tests
t1 <- t.test(SubLegStand ~ treat, data = harass[harass$treat %in% c(1, 2), ])
t2 <- t.test(SubLegStand ~ treat, data = harass[harass$treat %in% c(1, 3), ])
t3 <- t.test(ProLegStand ~ treat, data = harass[harass$treat %in% c(1, 2), ])
t4 <- t.test(ProLegStand ~ treat, data = harass[harass$treat %in% c(1, 3), ])

# Create results data frame
results <- data.frame(
  Variable = c("SubLegStand", "SubLegStand", "ProLegStand", "ProLegStand"),
  Comparison = c("All Male vs. Gender-Balanced",
                 "All Male vs. Quota Gender-Balanced",
                 "All Male vs. Gender-Balanced",
                 "All Male vs. Quota Gender-Balanced"),
  t_stat = c(t1$statistic, t2$statistic, t3$statistic, t4$statistic),
  p_value = c(t1$p.value, t2$p.value, t3$p.value, t4$p.value),
  CI_lower = c(t1$conf.int[1], t2$conf.int[1], t3$conf.int[1], t4$conf.int[1]),
  CI_upper = c(t1$conf.int[2], t2$conf.int[2], t3$conf.int[2], t4$conf.int[2])
)

rownames(results) <- c("T1: SubLegStand (1 vs. 2)",
                       "T2: SubLegStand (1 vs. 3)",
                       "T3: ProLegStand (1 vs. 2)",
                       "T4: ProLegStand (1 vs. 3)")

latex_table <- xtable(results)

sink(file.path(tables_dir, "tableSI.6.tex"))
print(latex_table, type = "latex")
sink()

cat("Table SI.6 saved.\n")

### TABLE SI.8: Treatment effects for US and UK ###

cat("Creating Table SI.8...\n")

USA <- subset(harass, Country == "USA")
UK <- subset(harass, Country == "UK")

# Perform t-tests for the US
t1 <- t.test(SubLegStand ~ treat, data = USA[USA$treat %in% c(1, 2), ])
t2 <- t.test(SubLegStand ~ treat, data = USA[USA$treat %in% c(1, 3), ])
t3 <- t.test(ProLegStand ~ treat, data = USA[USA$treat %in% c(1, 2), ])
t4 <- t.test(ProLegStand ~ treat, data = USA[USA$treat %in% c(1, 3), ])

# Perform t-tests for the UK
t5 <- t.test(SubLegStand ~ treat, data = UK[UK$treat %in% c(1, 2), ])
t6 <- t.test(SubLegStand ~ treat, data = UK[UK$treat %in% c(1, 3), ])
t7 <- t.test(ProLegStand ~ treat, data = UK[UK$treat %in% c(1, 2), ])
t8 <- t.test(ProLegStand ~ treat, data = UK[UK$treat %in% c(1, 3), ])

# Create results data frame
results <- data.frame(
  Country = c(rep("USA", 4), rep("UK", 4)),
  Variable = c(rep("SubLegStand", 2), rep("ProLegStand", 2),
               rep("SubLegStand", 2), rep("ProLegStand", 2)),
  Comparison = c("All Male vs. Gender-Balanced", "All Male vs. Quota Gender-Balanced",
                 "All Male vs. Gender-Balanced", "All Male vs. Quota Gender-Balanced",
                 "All Male vs. Gender-Balanced", "All Male vs. Quota Gender-Balanced",
                 "All Male vs. Gender-Balanced", "All Male vs. Quota Gender-Balanced"),
  t_stat = c(t1$statistic, t2$statistic, t3$statistic, t4$statistic,
             t5$statistic, t6$statistic, t7$statistic, t8$statistic),
  p_value = c(t1$p.value, t2$p.value, t3$p.value, t4$p.value,
              t5$p.value, t6$p.value, t7$p.value, t8$p.value),
  CI_lower = c(t1$conf.int[1], t2$conf.int[1], t3$conf.int[1], t4$conf.int[1],
               t5$conf.int[1], t6$conf.int[1], t7$conf.int[1], t8$conf.int[1]),
  CI_upper = c(t1$conf.int[2], t2$conf.int[2], t3$conf.int[2], t4$conf.int[2],
               t5$conf.int[2], t6$conf.int[2], t7$conf.int[2], t8$conf.int[2])
)

rownames(results) <- c("SI.8: USA - T1", "SI.8: USA - T2", "SI.8: USA - T3", "SI.8: USA - T4",
                       "SI.8: UK - T5", "SI.8: UK - T6", "SI.8: UK - T7", "SI.8: UK - T8")

latex_table <- xtable(results)

sink(file.path(tables_dir, "tableSI.8.tex"))
print(latex_table, type = "latex")
sink()

cat("Table SI.8 saved.\n")

### TABLE SI.9: Treatment effects for animal mistreatment ###

cat("Creating Table SI.9...\n")

# Perform t-tests for animal mistreatment vignettes
t1 <- t.test(SubLegStand ~ treat, data = animal[animal$treat %in% c(4, 5), ])
t2 <- t.test(SubLegStand ~ treat, data = animal[animal$treat %in% c(4, 6), ])
t3 <- t.test(ProLegStand ~ treat, data = animal[animal$treat %in% c(4, 5), ])
t4 <- t.test(ProLegStand ~ treat, data = animal[animal$treat %in% c(4, 6), ])

results <- data.frame(
  Variable = c(rep("SubLegStand", 2), rep("ProLegStand", 2)),
  Comparison = c("All Male vs. Gender-Balanced", "All Male vs. Quota Gender-Balanced",
                 "All Male vs. Gender-Balanced", "All Male vs. Quota Gender-Balanced"),
  t_stat = c(t1$statistic, t2$statistic, t3$statistic, t4$statistic),
  p_value = c(t1$p.value, t2$p.value, t3$p.value, t4$p.value),
  CI_lower = c(t1$conf.int[1], t2$conf.int[1], t3$conf.int[1], t4$conf.int[1]),
  CI_upper = c(t1$conf.int[2], t2$conf.int[2], t3$conf.int[2], t4$conf.int[2])
)

rownames(results) <- c("SI.9: Animal - T1", "SI.9: Animal - T2", "SI.9: Animal - T3", "SI.9: Animal - T4")

latex_table <- xtable(results)

sink(file.path(tables_dir, "tableSI9.tex"))
print(latex_table, type = "latex")
sink()

cat("Table SI.9 saved.\n")

### TABLE SI.10: Model-based specification with covariates - substantive legitimacy ###

cat("Creating Table SI.10...\n")

harassGB <- subset(harass, treat == 2 | treat == 3)
harassAM_GB <- subset(harass, treat == 1 | treat == 2)
harassAM_GBQ <- subset(harass, treat == 1 | treat == 3)

harassAM_GBQ$treat <- recode(harassAM_GBQ$treat, "3 = 2")

# Main models for outcome of substantive legitimacy
model1 <- lm(SubLegStand ~ treat + gender + leftright_1 + Country, data = harassAM_GB)
model2 <- lm(SubLegStand ~ treat + gender + leftright_1 + Country, data = harassAM_GBQ)
model3 <- lm(SubLegStand ~ treat + gender + leftright_1 + Country, data = harassGB)

# Extract coefficients
extract_coefficients <- function(model) {
  summary_model <- summary(model)
  coefs <- summary_model$coefficients[, 1:2]
  return(data.frame(Estimate = coefs[, 1], StdError = coefs[, 2]))
}

models <- list(model1, model2, model3)
model_results <- lapply(models, extract_coefficients)

coef_names <- rownames(model_results[[1]])
results_df <- data.frame(
  Coefficient = coef_names,
  Model1_Estimate = model_results[[1]]$Estimate,
  Model1_StdError = model_results[[1]]$StdError,
  Model2_Estimate = model_results[[2]]$Estimate,
  Model2_StdError = model_results[[2]]$StdError,
  Model3_Estimate = model_results[[3]]$Estimate,
  Model3_StdError = model_results[[3]]$StdError
)

latex_table <- xtable(results_df)

sink(file.path(tables_dir, "tableSI10.tex"))
print(latex_table, type = "latex")
sink()

cat("Table SI.10 saved.\n")

### TABLE SI.11: Model-based specification with covariates - procedural legitimacy ###

cat("Creating Table SI.11...\n")

model1 <- lm(ProLegStand ~ treat + gender + leftright_1 + Country, data = harassAM_GB)
model2 <- lm(ProLegStand ~ treat + gender + leftright_1 + Country, data = harassAM_GBQ)
model3 <- lm(ProLegStand ~ treat + gender + leftright_1 + Country, data = harassGB)

models <- list(model1, model2, model3)
model_results <- lapply(models, extract_coefficients)

coef_names <- rownames(model_results[[1]])
results_df <- data.frame(
  Coefficient = coef_names,
  Model1_Estimate = model_results[[1]]$Estimate,
  Model1_StdError = model_results[[1]]$StdError,
  Model2_Estimate = model_results[[2]]$Estimate,
  Model2_StdError = model_results[[2]]$StdError,
  Model3_Estimate = model_results[[3]]$Estimate,
  Model3_StdError = model_results[[3]]$StdError
)

latex_table <- xtable(results_df)

sink(file.path(tables_dir, "tableSI11.tex"))
print(latex_table, type = "latex")
sink()

cat("Table SI.11 saved.\n")

### TABLE SI.12: Robustness check excluding political left - substantive legitimacy ###

cat("Creating Table SI.12...\n")

# Excluding political liberals -- those that score five or below on the ideological self placement scale
harassAM_GB_ModCon <- subset(harassAM_GB, leftright_1 > 4)
harassAM_GBQ_ModCon <- subset(harassAM_GBQ, leftright_1 > 4)
harassGB_ModCon <- subset(harassGB, leftright_1 > 4)

model1 <- lm(SubLegStand ~ treat + gender + leftright_1 + Country, data = harassAM_GB_ModCon)
model2 <- lm(SubLegStand ~ treat + gender + leftright_1 + Country, data = harassAM_GBQ_ModCon)
model3 <- lm(SubLegStand ~ treat + gender + leftright_1 + Country, data = harassGB_ModCon)

models <- list(model1, model2, model3)
model_results <- lapply(models, extract_coefficients)

coef_names <- rownames(model_results[[1]])
results_df <- data.frame(
  Coefficient = coef_names,
  Model1_Estimate = model_results[[1]]$Estimate,
  Model1_StdError = model_results[[1]]$StdError,
  Model2_Estimate = model_results[[2]]$Estimate,
  Model2_StdError = model_results[[2]]$StdError,
  Model3_Estimate = model_results[[3]]$Estimate,
  Model3_StdError = model_results[[3]]$StdError
)

latex_table <- xtable(results_df)

sink(file.path(tables_dir, "tableSI12.tex"))
print(latex_table, type = "latex")
sink()

cat("Table SI.12 saved.\n")

### TABLE SI.13: Robustness check excluding political left - procedural legitimacy ###

cat("Creating Table SI.13...\n")

model1 <- lm(ProLegStand ~ treat + gender + leftright_1 + Country, data = harassAM_GB_ModCon)
model2 <- lm(ProLegStand ~ treat + gender + leftright_1 + Country, data = harassAM_GBQ_ModCon)
model3 <- lm(ProLegStand ~ treat + gender + leftright_1 + Country, data = harassGB_ModCon)

models <- list(model1, model2, model3)
model_results <- lapply(models, extract_coefficients)

coef_names <- rownames(model_results[[1]])
results_df <- data.frame(
  Coefficient = coef_names,
  Model1_Estimate = model_results[[1]]$Estimate,
  Model1_StdError = model_results[[1]]$StdError,
  Model2_Estimate = model_results[[2]]$Estimate,
  Model2_StdError = model_results[[2]]$StdError,
  Model3_Estimate = model_results[[3]]$Estimate,
  Model3_StdError = model_results[[3]]$StdError
)

latex_table <- xtable(results_df)

sink(file.path(tables_dir, "tableSI13.tex"))
print(latex_table, type = "latex")
sink()

cat("Table SI.13 saved.\n")

cat("All tables generated successfully.\n")


## 5. OLS Regressions with Covariates


### 5. OLS Regressions with Covariates

Model-based estimates of treatment effects controlling for covariates
and country fixed effects.

**Table SI.10:** Substantive legitimacy regressions
- Model 1: All-Male vs Gender-Balanced + controls
- Model 2: All-Male vs Quota Gender-Balanced + controls
- Model 3: Gender-Balanced vs Quota Gender-Balanced + controls

**Table SI.11:** Procedural legitimacy regressions
- Same models as SI.10 but for procedural outcome

Controls:
- Gender (respondent)
- Political ideology (leftright_1, 0-10 scale)
- Country fixed effects (11 dummies)

Formula: Outcome ~ treat + gender + leftright_1 + Country

In [ ]:
# Generate all supplementary tables

cat("Generating tables...\n")

# Reload harass and animal from merged data (in case modified by figures script)
harass <- subset(MergedN, treat == 1 | treat == 2 | treat == 3)
animal <- subset(MergedN, treat == 4 | treat == 5 | treat == 6)

### TABLE SI.4: Balance on women's issue treatments ###

cat("Creating Table SI.4...\n")

# Create country dummy variables
harass$Argentina <- recode(harass$Country, "'Argentina' = 1; else = 0")
harass$Australia <- recode(harass$Country, "'Australia' = 1; else = 0")
harass$Brazil <- recode(harass$Country, "'Brazil' = 1; else = 0")
harass$France <- recode(harass$Country, "'France' = 1; else = 0")
harass$Mexico <- recode(harass$Country, "'Mexico' = 1; else = 0")
harass$Norway <- recode(harass$Country, "'Norway' = 1; else = 0")
harass$NZ <- recode(harass$Country, "'NZ' = 1; else = 0")
harass$Peru <- recode(harass$Country, "'Peru' = 1; else = 0")
harass$Portugal <- recode(harass$Country, "'Portugal' = 1; else = 0")
harass$Spain <- recode(harass$Country, "'Spain' = 1; else = 0")
harass$UK <- recode(harass$Country, "'UK' = 1; else = 0")
harass$USA <- recode(harass$Country, "'USA' = 1; else = 0")

# Calculate means by treatment
tgender <- tapply(harass$gender, harass$treat, mean, na.rm = TRUE)
tideo <- tapply(harass$leftright_1, harass$treat, mean, na.rm = TRUE)
targ <- tapply(harass$Argentina, harass$treat, mean, na.rm = TRUE)
taus <- tapply(harass$Australia, harass$treat, mean, na.rm = TRUE)
tbra <- tapply(harass$Brazil, harass$treat, mean, na.rm = TRUE)
tfra <- tapply(harass$France, harass$treat, mean, na.rm = TRUE)
tmex <- tapply(harass$Mexico, harass$treat, mean, na.rm = TRUE)
tnor <- tapply(harass$Norway, harass$treat, mean, na.rm = TRUE)
tnz <- tapply(harass$NZ, harass$treat, mean, na.rm = TRUE)
tper <- tapply(harass$Peru, harass$treat, mean, na.rm = TRUE)
tpor <- tapply(harass$Portugal, harass$treat, mean, na.rm = TRUE)
tspa <- tapply(harass$Spain, harass$treat, mean, na.rm = TRUE)
tuk <- tapply(harass$UK, harass$treat, mean, na.rm = TRUE)
tusa <- tapply(harass$USA, harass$treat, mean, na.rm = TRUE)

tapply_df <- as.data.frame(rbind(tgender, tideo, targ, taus, tbra, tfra, tmex, tnor, tnz, tper, tpor, tspa, tuk, tusa))
latex_table <- xtable(tapply_df)

sink(file.path(tables_dir, "tableSI.4.tex"))
print(latex_table, type = "latex")
sink()

cat("Table SI.4 saved.\n")

### TABLE SI.5: Balance on animal treatment treatments ###

cat("Creating Table SI.5...\n")

# Create country dummy variables for animal
animal$Argentina <- recode(animal$Country, "'Argentina' = 1; else = 0")
animal$Australia <- recode(animal$Country, "'Australia' = 1; else = 0")
animal$Brazil <- recode(animal$Country, "'Brazil' = 1; else = 0")
animal$France <- recode(animal$Country, "'France' = 1; else = 0")
animal$Mexico <- recode(animal$Country, "'Mexico' = 1; else = 0")
animal$Norway <- recode(animal$Country, "'Norway' = 1; else = 0")
animal$NZ <- recode(animal$Country, "'NZ' = 1; else = 0")
animal$Peru <- recode(animal$Country, "'Peru' = 1; else = 0")
animal$Portugal <- recode(animal$Country, "'Portugal' = 1; else = 0")
animal$Spain <- recode(animal$Country, "'Spain' = 1; else = 0")
animal$UK <- recode(animal$Country, "'UK' = 1; else = 0")
animal$USA <- recode(animal$Country, "'USA' = 1; else = 0")

tgender <- tapply(animal$gender, animal$treat, mean, na.rm = TRUE)
tideo <- tapply(animal$leftright_1, animal$treat, mean, na.rm = TRUE)
targ <- tapply(animal$Argentina, animal$treat, mean, na.rm = TRUE)
taus <- tapply(animal$Australia, animal$treat, mean, na.rm = TRUE)
tbra <- tapply(animal$Brazil, animal$treat, mean, na.rm = TRUE)
tfra <- tapply(animal$France, animal$treat, mean, na.rm = TRUE)
tmex <- tapply(animal$Mexico, animal$treat, mean, na.rm = TRUE)
tnor <- tapply(animal$Norway, animal$treat, mean, na.rm = TRUE)
tnz <- tapply(animal$NZ, animal$treat, mean, na.rm = TRUE)
tper <- tapply(animal$Peru, animal$treat, mean, na.rm = TRUE)
tpor <- tapply(animal$Portugal, animal$treat, mean, na.rm = TRUE)
tspa <- tapply(animal$Spain, animal$treat, mean, na.rm = TRUE)
tuk <- tapply(animal$UK, animal$treat, mean, na.rm = TRUE)
tusa <- tapply(animal$USA, animal$treat, mean, na.rm = TRUE)

tapply_df <- as.data.frame(rbind(tgender, tideo, targ, taus, tbra, tfra, tmex, tnor, tnz, tper, tpor, tspa, tuk, tusa))
latex_table <- xtable(tapply_df)

sink(file.path(tables_dir, "tableSI.5.tex"))
print(latex_table, type = "latex")
sink()

cat("Table SI.5 saved.\n")

### TABLE SI.6: Main cross-country treatment effects ###

cat("Creating Table SI.6...\n")

# Perform t-tests
t1 <- t.test(SubLegStand ~ treat, data = harass[harass$treat %in% c(1, 2), ])
t2 <- t.test(SubLegStand ~ treat, data = harass[harass$treat %in% c(1, 3), ])
t3 <- t.test(ProLegStand ~ treat, data = harass[harass$treat %in% c(1, 2), ])
t4 <- t.test(ProLegStand ~ treat, data = harass[harass$treat %in% c(1, 3), ])

# Create results data frame
results <- data.frame(
  Variable = c("SubLegStand", "SubLegStand", "ProLegStand", "ProLegStand"),
  Comparison = c("All Male vs. Gender-Balanced",
                 "All Male vs. Quota Gender-Balanced",
                 "All Male vs. Gender-Balanced",
                 "All Male vs. Quota Gender-Balanced"),
  t_stat = c(t1$statistic, t2$statistic, t3$statistic, t4$statistic),
  p_value = c(t1$p.value, t2$p.value, t3$p.value, t4$p.value),
  CI_lower = c(t1$conf.int[1], t2$conf.int[1], t3$conf.int[1], t4$conf.int[1]),
  CI_upper = c(t1$conf.int[2], t2$conf.int[2], t3$conf.int[2], t4$conf.int[2])
)

rownames(results) <- c("T1: SubLegStand (1 vs. 2)",
                       "T2: SubLegStand (1 vs. 3)",
                       "T3: ProLegStand (1 vs. 2)",
                       "T4: ProLegStand (1 vs. 3)")

latex_table <- xtable(results)

sink(file.path(tables_dir, "tableSI.6.tex"))
print(latex_table, type = "latex")
sink()

cat("Table SI.6 saved.\n")

### TABLE SI.8: Treatment effects for US and UK ###

cat("Creating Table SI.8...\n")

USA <- subset(harass, Country == "USA")
UK <- subset(harass, Country == "UK")

# Perform t-tests for the US
t1 <- t.test(SubLegStand ~ treat, data = USA[USA$treat %in% c(1, 2), ])
t2 <- t.test(SubLegStand ~ treat, data = USA[USA$treat %in% c(1, 3), ])
t3 <- t.test(ProLegStand ~ treat, data = USA[USA$treat %in% c(1, 2), ])
t4 <- t.test(ProLegStand ~ treat, data = USA[USA$treat %in% c(1, 3), ])

# Perform t-tests for the UK
t5 <- t.test(SubLegStand ~ treat, data = UK[UK$treat %in% c(1, 2), ])
t6 <- t.test(SubLegStand ~ treat, data = UK[UK$treat %in% c(1, 3), ])
t7 <- t.test(ProLegStand ~ treat, data = UK[UK$treat %in% c(1, 2), ])
t8 <- t.test(ProLegStand ~ treat, data = UK[UK$treat %in% c(1, 3), ])

# Create results data frame
results <- data.frame(
  Country = c(rep("USA", 4), rep("UK", 4)),
  Variable = c(rep("SubLegStand", 2), rep("ProLegStand", 2),
               rep("SubLegStand", 2), rep("ProLegStand", 2)),
  Comparison = c("All Male vs. Gender-Balanced", "All Male vs. Quota Gender-Balanced",
                 "All Male vs. Gender-Balanced", "All Male vs. Quota Gender-Balanced",
                 "All Male vs. Gender-Balanced", "All Male vs. Quota Gender-Balanced",
                 "All Male vs. Gender-Balanced", "All Male vs. Quota Gender-Balanced"),
  t_stat = c(t1$statistic, t2$statistic, t3$statistic, t4$statistic,
             t5$statistic, t6$statistic, t7$statistic, t8$statistic),
  p_value = c(t1$p.value, t2$p.value, t3$p.value, t4$p.value,
              t5$p.value, t6$p.value, t7$p.value, t8$p.value),
  CI_lower = c(t1$conf.int[1], t2$conf.int[1], t3$conf.int[1], t4$conf.int[1],
               t5$conf.int[1], t6$conf.int[1], t7$conf.int[1], t8$conf.int[1]),
  CI_upper = c(t1$conf.int[2], t2$conf.int[2], t3$conf.int[2], t4$conf.int[2],
               t5$conf.int[2], t6$conf.int[2], t7$conf.int[2], t8$conf.int[2])
)

rownames(results) <- c("SI.8: USA - T1", "SI.8: USA - T2", "SI.8: USA - T3", "SI.8: USA - T4",
                       "SI.8: UK - T5", "SI.8: UK - T6", "SI.8: UK - T7", "SI.8: UK - T8")

latex_table <- xtable(results)

sink(file.path(tables_dir, "tableSI.8.tex"))
print(latex_table, type = "latex")
sink()

cat("Table SI.8 saved.\n")

### TABLE SI.9: Treatment effects for animal mistreatment ###

cat("Creating Table SI.9...\n")

# Perform t-tests for animal mistreatment vignettes
t1 <- t.test(SubLegStand ~ treat, data = animal[animal$treat %in% c(4, 5), ])
t2 <- t.test(SubLegStand ~ treat, data = animal[animal$treat %in% c(4, 6), ])
t3 <- t.test(ProLegStand ~ treat, data = animal[animal$treat %in% c(4, 5), ])
t4 <- t.test(ProLegStand ~ treat, data = animal[animal$treat %in% c(4, 6), ])

results <- data.frame(
  Variable = c(rep("SubLegStand", 2), rep("ProLegStand", 2)),
  Comparison = c("All Male vs. Gender-Balanced", "All Male vs. Quota Gender-Balanced",
                 "All Male vs. Gender-Balanced", "All Male vs. Quota Gender-Balanced"),
  t_stat = c(t1$statistic, t2$statistic, t3$statistic, t4$statistic),
  p_value = c(t1$p.value, t2$p.value, t3$p.value, t4$p.value),
  CI_lower = c(t1$conf.int[1], t2$conf.int[1], t3$conf.int[1], t4$conf.int[1]),
  CI_upper = c(t1$conf.int[2], t2$conf.int[2], t3$conf.int[2], t4$conf.int[2])
)

rownames(results) <- c("SI.9: Animal - T1", "SI.9: Animal - T2", "SI.9: Animal - T3", "SI.9: Animal - T4")

latex_table <- xtable(results)

sink(file.path(tables_dir, "tableSI9.tex"))
print(latex_table, type = "latex")
sink()

cat("Table SI.9 saved.\n")

### TABLE SI.10: Model-based specification with covariates - substantive legitimacy ###

cat("Creating Table SI.10...\n")

harassGB <- subset(harass, treat == 2 | treat == 3)
harassAM_GB <- subset(harass, treat == 1 | treat == 2)
harassAM_GBQ <- subset(harass, treat == 1 | treat == 3)

harassAM_GBQ$treat <- recode(harassAM_GBQ$treat, "3 = 2")

# Main models for outcome of substantive legitimacy
model1 <- lm(SubLegStand ~ treat + gender + leftright_1 + Country, data = harassAM_GB)
model2 <- lm(SubLegStand ~ treat + gender + leftright_1 + Country, data = harassAM_GBQ)
model3 <- lm(SubLegStand ~ treat + gender + leftright_1 + Country, data = harassGB)

# Extract coefficients
extract_coefficients <- function(model) {
  summary_model <- summary(model)
  coefs <- summary_model$coefficients[, 1:2]
  return(data.frame(Estimate = coefs[, 1], StdError = coefs[, 2]))
}

models <- list(model1, model2, model3)
model_results <- lapply(models, extract_coefficients)

coef_names <- rownames(model_results[[1]])
results_df <- data.frame(
  Coefficient = coef_names,
  Model1_Estimate = model_results[[1]]$Estimate,
  Model1_StdError = model_results[[1]]$StdError,
  Model2_Estimate = model_results[[2]]$Estimate,
  Model2_StdError = model_results[[2]]$StdError,
  Model3_Estimate = model_results[[3]]$Estimate,
  Model3_StdError = model_results[[3]]$StdError
)

latex_table <- xtable(results_df)

sink(file.path(tables_dir, "tableSI10.tex"))
print(latex_table, type = "latex")
sink()

cat("Table SI.10 saved.\n")

### TABLE SI.11: Model-based specification with covariates - procedural legitimacy ###

cat("Creating Table SI.11...\n")

model1 <- lm(ProLegStand ~ treat + gender + leftright_1 + Country, data = harassAM_GB)
model2 <- lm(ProLegStand ~ treat + gender + leftright_1 + Country, data = harassAM_GBQ)
model3 <- lm(ProLegStand ~ treat + gender + leftright_1 + Country, data = harassGB)

models <- list(model1, model2, model3)
model_results <- lapply(models, extract_coefficients)

coef_names <- rownames(model_results[[1]])
results_df <- data.frame(
  Coefficient = coef_names,
  Model1_Estimate = model_results[[1]]$Estimate,
  Model1_StdError = model_results[[1]]$StdError,
  Model2_Estimate = model_results[[2]]$Estimate,
  Model2_StdError = model_results[[2]]$StdError,
  Model3_Estimate = model_results[[3]]$Estimate,
  Model3_StdError = model_results[[3]]$StdError
)

latex_table <- xtable(results_df)

sink(file.path(tables_dir, "tableSI11.tex"))
print(latex_table, type = "latex")
sink()

cat("Table SI.11 saved.\n")

### TABLE SI.12: Robustness check excluding political left - substantive legitimacy ###

cat("Creating Table SI.12...\n")

# Excluding political liberals -- those that score five or below on the ideological self placement scale
harassAM_GB_ModCon <- subset(harassAM_GB, leftright_1 > 4)
harassAM_GBQ_ModCon <- subset(harassAM_GBQ, leftright_1 > 4)
harassGB_ModCon <- subset(harassGB, leftright_1 > 4)

model1 <- lm(SubLegStand ~ treat + gender + leftright_1 + Country, data = harassAM_GB_ModCon)
model2 <- lm(SubLegStand ~ treat + gender + leftright_1 + Country, data = harassAM_GBQ_ModCon)
model3 <- lm(SubLegStand ~ treat + gender + leftright_1 + Country, data = harassGB_ModCon)

models <- list(model1, model2, model3)
model_results <- lapply(models, extract_coefficients)

coef_names <- rownames(model_results[[1]])
results_df <- data.frame(
  Coefficient = coef_names,
  Model1_Estimate = model_results[[1]]$Estimate,
  Model1_StdError = model_results[[1]]$StdError,
  Model2_Estimate = model_results[[2]]$Estimate,
  Model2_StdError = model_results[[2]]$StdError,
  Model3_Estimate = model_results[[3]]$Estimate,
  Model3_StdError = model_results[[3]]$StdError
)

latex_table <- xtable(results_df)

sink(file.path(tables_dir, "tableSI12.tex"))
print(latex_table, type = "latex")
sink()

cat("Table SI.12 saved.\n")

### TABLE SI.13: Robustness check excluding political left - procedural legitimacy ###

cat("Creating Table SI.13...\n")

model1 <- lm(ProLegStand ~ treat + gender + leftright_1 + Country, data = harassAM_GB_ModCon)
model2 <- lm(ProLegStand ~ treat + gender + leftright_1 + Country, data = harassAM_GBQ_ModCon)
model3 <- lm(ProLegStand ~ treat + gender + leftright_1 + Country, data = harassGB_ModCon)

models <- list(model1, model2, model3)
model_results <- lapply(models, extract_coefficients)

coef_names <- rownames(model_results[[1]])
results_df <- data.frame(
  Coefficient = coef_names,
  Model1_Estimate = model_results[[1]]$Estimate,
  Model1_StdError = model_results[[1]]$StdError,
  Model2_Estimate = model_results[[2]]$Estimate,
  Model2_StdError = model_results[[2]]$StdError,
  Model3_Estimate = model_results[[3]]$Estimate,
  Model3_StdError = model_results[[3]]$StdError
)

latex_table <- xtable(results_df)

sink(file.path(tables_dir, "tableSI13.tex"))
print(latex_table, type = "latex")
sink()

cat("Table SI.13 saved.\n")

cat("All tables generated successfully.\n")


## 6. Robustness Checks (Excluding Political Left)


### 6. Robustness Checks (Excluding Political Left)

Test whether results hold when excluding liberal respondents
(ideology ≤ 4 on 0-10 scale).

**Table SI.12:** Substantive legitimacy (moderates/conservatives only)
- Same three models as SI.10
- Subset: leftright_1 > 4
- Expected: Effects attenuated but still significant

**Table SI.13:** Procedural legitimacy (moderates/conservatives only)
- Same three models as SI.11
- Subset: leftright_1 > 4

Purpose: Address concern that quota support driven by left-leaning
respondents only.

In [ ]:
# Generate all supplementary tables

cat("Generating tables...\n")

# Reload harass and animal from merged data (in case modified by figures script)
harass <- subset(MergedN, treat == 1 | treat == 2 | treat == 3)
animal <- subset(MergedN, treat == 4 | treat == 5 | treat == 6)

### TABLE SI.4: Balance on women's issue treatments ###

cat("Creating Table SI.4...\n")

# Create country dummy variables
harass$Argentina <- recode(harass$Country, "'Argentina' = 1; else = 0")
harass$Australia <- recode(harass$Country, "'Australia' = 1; else = 0")
harass$Brazil <- recode(harass$Country, "'Brazil' = 1; else = 0")
harass$France <- recode(harass$Country, "'France' = 1; else = 0")
harass$Mexico <- recode(harass$Country, "'Mexico' = 1; else = 0")
harass$Norway <- recode(harass$Country, "'Norway' = 1; else = 0")
harass$NZ <- recode(harass$Country, "'NZ' = 1; else = 0")
harass$Peru <- recode(harass$Country, "'Peru' = 1; else = 0")
harass$Portugal <- recode(harass$Country, "'Portugal' = 1; else = 0")
harass$Spain <- recode(harass$Country, "'Spain' = 1; else = 0")
harass$UK <- recode(harass$Country, "'UK' = 1; else = 0")
harass$USA <- recode(harass$Country, "'USA' = 1; else = 0")

# Calculate means by treatment
tgender <- tapply(harass$gender, harass$treat, mean, na.rm = TRUE)
tideo <- tapply(harass$leftright_1, harass$treat, mean, na.rm = TRUE)
targ <- tapply(harass$Argentina, harass$treat, mean, na.rm = TRUE)
taus <- tapply(harass$Australia, harass$treat, mean, na.rm = TRUE)
tbra <- tapply(harass$Brazil, harass$treat, mean, na.rm = TRUE)
tfra <- tapply(harass$France, harass$treat, mean, na.rm = TRUE)
tmex <- tapply(harass$Mexico, harass$treat, mean, na.rm = TRUE)
tnor <- tapply(harass$Norway, harass$treat, mean, na.rm = TRUE)
tnz <- tapply(harass$NZ, harass$treat, mean, na.rm = TRUE)
tper <- tapply(harass$Peru, harass$treat, mean, na.rm = TRUE)
tpor <- tapply(harass$Portugal, harass$treat, mean, na.rm = TRUE)
tspa <- tapply(harass$Spain, harass$treat, mean, na.rm = TRUE)
tuk <- tapply(harass$UK, harass$treat, mean, na.rm = TRUE)
tusa <- tapply(harass$USA, harass$treat, mean, na.rm = TRUE)

tapply_df <- as.data.frame(rbind(tgender, tideo, targ, taus, tbra, tfra, tmex, tnor, tnz, tper, tpor, tspa, tuk, tusa))
latex_table <- xtable(tapply_df)

sink(file.path(tables_dir, "tableSI.4.tex"))
print(latex_table, type = "latex")
sink()

cat("Table SI.4 saved.\n")

### TABLE SI.5: Balance on animal treatment treatments ###

cat("Creating Table SI.5...\n")

# Create country dummy variables for animal
animal$Argentina <- recode(animal$Country, "'Argentina' = 1; else = 0")
animal$Australia <- recode(animal$Country, "'Australia' = 1; else = 0")
animal$Brazil <- recode(animal$Country, "'Brazil' = 1; else = 0")
animal$France <- recode(animal$Country, "'France' = 1; else = 0")
animal$Mexico <- recode(animal$Country, "'Mexico' = 1; else = 0")
animal$Norway <- recode(animal$Country, "'Norway' = 1; else = 0")
animal$NZ <- recode(animal$Country, "'NZ' = 1; else = 0")
animal$Peru <- recode(animal$Country, "'Peru' = 1; else = 0")
animal$Portugal <- recode(animal$Country, "'Portugal' = 1; else = 0")
animal$Spain <- recode(animal$Country, "'Spain' = 1; else = 0")
animal$UK <- recode(animal$Country, "'UK' = 1; else = 0")
animal$USA <- recode(animal$Country, "'USA' = 1; else = 0")

tgender <- tapply(animal$gender, animal$treat, mean, na.rm = TRUE)
tideo <- tapply(animal$leftright_1, animal$treat, mean, na.rm = TRUE)
targ <- tapply(animal$Argentina, animal$treat, mean, na.rm = TRUE)
taus <- tapply(animal$Australia, animal$treat, mean, na.rm = TRUE)
tbra <- tapply(animal$Brazil, animal$treat, mean, na.rm = TRUE)
tfra <- tapply(animal$France, animal$treat, mean, na.rm = TRUE)
tmex <- tapply(animal$Mexico, animal$treat, mean, na.rm = TRUE)
tnor <- tapply(animal$Norway, animal$treat, mean, na.rm = TRUE)
tnz <- tapply(animal$NZ, animal$treat, mean, na.rm = TRUE)
tper <- tapply(animal$Peru, animal$treat, mean, na.rm = TRUE)
tpor <- tapply(animal$Portugal, animal$treat, mean, na.rm = TRUE)
tspa <- tapply(animal$Spain, animal$treat, mean, na.rm = TRUE)
tuk <- tapply(animal$UK, animal$treat, mean, na.rm = TRUE)
tusa <- tapply(animal$USA, animal$treat, mean, na.rm = TRUE)

tapply_df <- as.data.frame(rbind(tgender, tideo, targ, taus, tbra, tfra, tmex, tnor, tnz, tper, tpor, tspa, tuk, tusa))
latex_table <- xtable(tapply_df)

sink(file.path(tables_dir, "tableSI.5.tex"))
print(latex_table, type = "latex")
sink()

cat("Table SI.5 saved.\n")

### TABLE SI.6: Main cross-country treatment effects ###

cat("Creating Table SI.6...\n")

# Perform t-tests
t1 <- t.test(SubLegStand ~ treat, data = harass[harass$treat %in% c(1, 2), ])
t2 <- t.test(SubLegStand ~ treat, data = harass[harass$treat %in% c(1, 3), ])
t3 <- t.test(ProLegStand ~ treat, data = harass[harass$treat %in% c(1, 2), ])
t4 <- t.test(ProLegStand ~ treat, data = harass[harass$treat %in% c(1, 3), ])

# Create results data frame
results <- data.frame(
  Variable = c("SubLegStand", "SubLegStand", "ProLegStand", "ProLegStand"),
  Comparison = c("All Male vs. Gender-Balanced",
                 "All Male vs. Quota Gender-Balanced",
                 "All Male vs. Gender-Balanced",
                 "All Male vs. Quota Gender-Balanced"),
  t_stat = c(t1$statistic, t2$statistic, t3$statistic, t4$statistic),
  p_value = c(t1$p.value, t2$p.value, t3$p.value, t4$p.value),
  CI_lower = c(t1$conf.int[1], t2$conf.int[1], t3$conf.int[1], t4$conf.int[1]),
  CI_upper = c(t1$conf.int[2], t2$conf.int[2], t3$conf.int[2], t4$conf.int[2])
)

rownames(results) <- c("T1: SubLegStand (1 vs. 2)",
                       "T2: SubLegStand (1 vs. 3)",
                       "T3: ProLegStand (1 vs. 2)",
                       "T4: ProLegStand (1 vs. 3)")

latex_table <- xtable(results)

sink(file.path(tables_dir, "tableSI.6.tex"))
print(latex_table, type = "latex")
sink()

cat("Table SI.6 saved.\n")

### TABLE SI.8: Treatment effects for US and UK ###

cat("Creating Table SI.8...\n")

USA <- subset(harass, Country == "USA")
UK <- subset(harass, Country == "UK")

# Perform t-tests for the US
t1 <- t.test(SubLegStand ~ treat, data = USA[USA$treat %in% c(1, 2), ])
t2 <- t.test(SubLegStand ~ treat, data = USA[USA$treat %in% c(1, 3), ])
t3 <- t.test(ProLegStand ~ treat, data = USA[USA$treat %in% c(1, 2), ])
t4 <- t.test(ProLegStand ~ treat, data = USA[USA$treat %in% c(1, 3), ])

# Perform t-tests for the UK
t5 <- t.test(SubLegStand ~ treat, data = UK[UK$treat %in% c(1, 2), ])
t6 <- t.test(SubLegStand ~ treat, data = UK[UK$treat %in% c(1, 3), ])
t7 <- t.test(ProLegStand ~ treat, data = UK[UK$treat %in% c(1, 2), ])
t8 <- t.test(ProLegStand ~ treat, data = UK[UK$treat %in% c(1, 3), ])

# Create results data frame
results <- data.frame(
  Country = c(rep("USA", 4), rep("UK", 4)),
  Variable = c(rep("SubLegStand", 2), rep("ProLegStand", 2),
               rep("SubLegStand", 2), rep("ProLegStand", 2)),
  Comparison = c("All Male vs. Gender-Balanced", "All Male vs. Quota Gender-Balanced",
                 "All Male vs. Gender-Balanced", "All Male vs. Quota Gender-Balanced",
                 "All Male vs. Gender-Balanced", "All Male vs. Quota Gender-Balanced",
                 "All Male vs. Gender-Balanced", "All Male vs. Quota Gender-Balanced"),
  t_stat = c(t1$statistic, t2$statistic, t3$statistic, t4$statistic,
             t5$statistic, t6$statistic, t7$statistic, t8$statistic),
  p_value = c(t1$p.value, t2$p.value, t3$p.value, t4$p.value,
              t5$p.value, t6$p.value, t7$p.value, t8$p.value),
  CI_lower = c(t1$conf.int[1], t2$conf.int[1], t3$conf.int[1], t4$conf.int[1],
               t5$conf.int[1], t6$conf.int[1], t7$conf.int[1], t8$conf.int[1]),
  CI_upper = c(t1$conf.int[2], t2$conf.int[2], t3$conf.int[2], t4$conf.int[2],
               t5$conf.int[2], t6$conf.int[2], t7$conf.int[2], t8$conf.int[2])
)

rownames(results) <- c("SI.8: USA - T1", "SI.8: USA - T2", "SI.8: USA - T3", "SI.8: USA - T4",
                       "SI.8: UK - T5", "SI.8: UK - T6", "SI.8: UK - T7", "SI.8: UK - T8")

latex_table <- xtable(results)

sink(file.path(tables_dir, "tableSI.8.tex"))
print(latex_table, type = "latex")
sink()

cat("Table SI.8 saved.\n")

### TABLE SI.9: Treatment effects for animal mistreatment ###

cat("Creating Table SI.9...\n")

# Perform t-tests for animal mistreatment vignettes
t1 <- t.test(SubLegStand ~ treat, data = animal[animal$treat %in% c(4, 5), ])
t2 <- t.test(SubLegStand ~ treat, data = animal[animal$treat %in% c(4, 6), ])
t3 <- t.test(ProLegStand ~ treat, data = animal[animal$treat %in% c(4, 5), ])
t4 <- t.test(ProLegStand ~ treat, data = animal[animal$treat %in% c(4, 6), ])

results <- data.frame(
  Variable = c(rep("SubLegStand", 2), rep("ProLegStand", 2)),
  Comparison = c("All Male vs. Gender-Balanced", "All Male vs. Quota Gender-Balanced",
                 "All Male vs. Gender-Balanced", "All Male vs. Quota Gender-Balanced"),
  t_stat = c(t1$statistic, t2$statistic, t3$statistic, t4$statistic),
  p_value = c(t1$p.value, t2$p.value, t3$p.value, t4$p.value),
  CI_lower = c(t1$conf.int[1], t2$conf.int[1], t3$conf.int[1], t4$conf.int[1]),
  CI_upper = c(t1$conf.int[2], t2$conf.int[2], t3$conf.int[2], t4$conf.int[2])
)

rownames(results) <- c("SI.9: Animal - T1", "SI.9: Animal - T2", "SI.9: Animal - T3", "SI.9: Animal - T4")

latex_table <- xtable(results)

sink(file.path(tables_dir, "tableSI9.tex"))
print(latex_table, type = "latex")
sink()

cat("Table SI.9 saved.\n")

### TABLE SI.10: Model-based specification with covariates - substantive legitimacy ###

cat("Creating Table SI.10...\n")

harassGB <- subset(harass, treat == 2 | treat == 3)
harassAM_GB <- subset(harass, treat == 1 | treat == 2)
harassAM_GBQ <- subset(harass, treat == 1 | treat == 3)

harassAM_GBQ$treat <- recode(harassAM_GBQ$treat, "3 = 2")

# Main models for outcome of substantive legitimacy
model1 <- lm(SubLegStand ~ treat + gender + leftright_1 + Country, data = harassAM_GB)
model2 <- lm(SubLegStand ~ treat + gender + leftright_1 + Country, data = harassAM_GBQ)
model3 <- lm(SubLegStand ~ treat + gender + leftright_1 + Country, data = harassGB)

# Extract coefficients
extract_coefficients <- function(model) {
  summary_model <- summary(model)
  coefs <- summary_model$coefficients[, 1:2]
  return(data.frame(Estimate = coefs[, 1], StdError = coefs[, 2]))
}

models <- list(model1, model2, model3)
model_results <- lapply(models, extract_coefficients)

coef_names <- rownames(model_results[[1]])
results_df <- data.frame(
  Coefficient = coef_names,
  Model1_Estimate = model_results[[1]]$Estimate,
  Model1_StdError = model_results[[1]]$StdError,
  Model2_Estimate = model_results[[2]]$Estimate,
  Model2_StdError = model_results[[2]]$StdError,
  Model3_Estimate = model_results[[3]]$Estimate,
  Model3_StdError = model_results[[3]]$StdError
)

latex_table <- xtable(results_df)

sink(file.path(tables_dir, "tableSI10.tex"))
print(latex_table, type = "latex")
sink()

cat("Table SI.10 saved.\n")

### TABLE SI.11: Model-based specification with covariates - procedural legitimacy ###

cat("Creating Table SI.11...\n")

model1 <- lm(ProLegStand ~ treat + gender + leftright_1 + Country, data = harassAM_GB)
model2 <- lm(ProLegStand ~ treat + gender + leftright_1 + Country, data = harassAM_GBQ)
model3 <- lm(ProLegStand ~ treat + gender + leftright_1 + Country, data = harassGB)

models <- list(model1, model2, model3)
model_results <- lapply(models, extract_coefficients)

coef_names <- rownames(model_results[[1]])
results_df <- data.frame(
  Coefficient = coef_names,
  Model1_Estimate = model_results[[1]]$Estimate,
  Model1_StdError = model_results[[1]]$StdError,
  Model2_Estimate = model_results[[2]]$Estimate,
  Model2_StdError = model_results[[2]]$StdError,
  Model3_Estimate = model_results[[3]]$Estimate,
  Model3_StdError = model_results[[3]]$StdError
)

latex_table <- xtable(results_df)

sink(file.path(tables_dir, "tableSI11.tex"))
print(latex_table, type = "latex")
sink()

cat("Table SI.11 saved.\n")

### TABLE SI.12: Robustness check excluding political left - substantive legitimacy ###

cat("Creating Table SI.12...\n")

# Excluding political liberals -- those that score five or below on the ideological self placement scale
harassAM_GB_ModCon <- subset(harassAM_GB, leftright_1 > 4)
harassAM_GBQ_ModCon <- subset(harassAM_GBQ, leftright_1 > 4)
harassGB_ModCon <- subset(harassGB, leftright_1 > 4)

model1 <- lm(SubLegStand ~ treat + gender + leftright_1 + Country, data = harassAM_GB_ModCon)
model2 <- lm(SubLegStand ~ treat + gender + leftright_1 + Country, data = harassAM_GBQ_ModCon)
model3 <- lm(SubLegStand ~ treat + gender + leftright_1 + Country, data = harassGB_ModCon)

models <- list(model1, model2, model3)
model_results <- lapply(models, extract_coefficients)

coef_names <- rownames(model_results[[1]])
results_df <- data.frame(
  Coefficient = coef_names,
  Model1_Estimate = model_results[[1]]$Estimate,
  Model1_StdError = model_results[[1]]$StdError,
  Model2_Estimate = model_results[[2]]$Estimate,
  Model2_StdError = model_results[[2]]$StdError,
  Model3_Estimate = model_results[[3]]$Estimate,
  Model3_StdError = model_results[[3]]$StdError
)

latex_table <- xtable(results_df)

sink(file.path(tables_dir, "tableSI12.tex"))
print(latex_table, type = "latex")
sink()

cat("Table SI.12 saved.\n")

### TABLE SI.13: Robustness check excluding political left - procedural legitimacy ###

cat("Creating Table SI.13...\n")

model1 <- lm(ProLegStand ~ treat + gender + leftright_1 + Country, data = harassAM_GB_ModCon)
model2 <- lm(ProLegStand ~ treat + gender + leftright_1 + Country, data = harassAM_GBQ_ModCon)
model3 <- lm(ProLegStand ~ treat + gender + leftright_1 + Country, data = harassGB_ModCon)

models <- list(model1, model2, model3)
model_results <- lapply(models, extract_coefficients)

coef_names <- rownames(model_results[[1]])
results_df <- data.frame(
  Coefficient = coef_names,
  Model1_Estimate = model_results[[1]]$Estimate,
  Model1_StdError = model_results[[1]]$StdError,
  Model2_Estimate = model_results[[2]]$Estimate,
  Model2_StdError = model_results[[2]]$StdError,
  Model3_Estimate = model_results[[3]]$Estimate,
  Model3_StdError = model_results[[3]]$StdError
)

latex_table <- xtable(results_df)

sink(file.path(tables_dir, "tableSI13.tex"))
print(latex_table, type = "latex")
sink()

cat("Table SI.13 saved.\n")

cat("All tables generated successfully.\n")
